## Combine pacbio maps

This script would not be necessary for satseq implementation in general

But for this paper, due to refining of methods, we ended up with several pacbio barcode-variant maps. This code was used to combine them into a map for each cell type, that was then used for 10x processing.

We have two separate maps for each cell type (4T1 and EO771.lmb), and also a plasmid map. The general rules are:
- We create a combined map for each of 4T1 and EO771.
- For 4T1, we combine 4T1_run1, 4T1_run2, and plasmid map
- For EO771, we combine EO771_run1, EO771_run2, and plasmid map
- We only use barcodes found in the plasmid map
- When the celltype-specific map has a different variant than the plasmid map, we use the celltype-specific one
- If neither celltype-specific map has observed a barcode, use the plasmid variant
- If the two celltype-specific maps have a barcode and disagree, but agree on position, convert to a position-specific variant (i.e., A261X)
- Otherwise resovle based on two-out-of-three agreeing, or better score

In [1]:
import os
import sys
from importlib.metadata import version

import numpy as np
import pandas as pd

In [2]:
print(f"Python: {sys.version.split()[0]}")

packages = [
    "matplotlib",
    "numpy",
    "pandas",
    "seaborn",
]

for package in packages:
    print(f"{package}: {version(package)}")

Python: 3.10.12
matplotlib: 3.8.3
numpy: 1.26.4
pandas: 2.2.1
seaborn: 0.13.2


In [3]:
# Print working directory for reference
os.getcwd()

'/workdir/satseq/satseq_processing'

In [4]:
output_dir='/workdir/satseq/long_read/bcmaps_combined'
!mkdir -p {output_dir}

In [5]:
bcmaps = {}
bcmaps['plasmid'] = pd.read_csv('/workdir/satseq/long_read/plasmid_20240927_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')
# we don't end up using this one
#bcmaps['plasmid_amplified'] = pd.read_csv('/workdir/satseq/long_read/plasmid_amplified_20250121_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')
bcmaps['EO771_run1'] = pd.read_csv('/workdir/satseq/long_read/EO771_20241220_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')
bcmaps['EO771_run2'] = pd.read_csv('/workdir/satseq/long_read/EO771_20250121_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')
bcmaps['4T1_run1'] = pd.read_csv('/workdir/satseq/long_read/4T1_20241218_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')
bcmaps['4T1_run2'] = pd.read_csv('/workdir/satseq/long_read/4T1_20250121_out/bcmap_2025_02_26.txt.gz', index_col=0, sep='\t')

In [6]:
variants = pd.read_csv('data/twist_mSTING_full_feature_ref_v3.csv', index_col=0).set_index('name', drop=True)
# assign a position, wt AA, and mutant AA to each variant. Most are indicated in variant name,
# but some special ones need values assigned. Use negative positions for non-positional variants

variants['pos'] = -1
variants['wt_aa'] = ''
variants['mut_aa'] = ''
for x in variants.index:
    wtaa=''
    mutaa=''
    pos=-100
    if x == 'STOP':
        pos = -2
    elif x == 'mSTING_WT':
        pos = 0
    elif x == 'dUPR':
        pos = -1
    elif x == 'dCTT':
        pos = -3
    elif x == 'RRDD':
        pos = -4
    elif x == 'RRAA':
        pos = -5
    else:
        if x[len(x)-1] == 'X' or x=='V146L' or x == 'N153S' or x=='Y239S' or x == 'S365A' or x == 'V154M' or x=='S161A' or x=='I229A' or x=='N241A':
            tmp = int(x[1:(len(x)-1)])
            pos = tmp
            wtaa=x[0]
            mutaa=x[len(x)-1]
        else:
            print(f'Error {x}')
    variants.loc[x,'pos'] = pos
    variants.loc[x,'wt_aa'] = wtaa
    variants.loc[x,'mut_aa'] = mutaa

In [7]:
nonPositionalVars = list(variants.loc[variants['pos'] < 1].index)
nonPositionalVars

['mSTING_WT', 'STOP', 'RRAA', 'RRDD', 'dCTT', 'dUPR']

In [8]:
for k,bcmap in bcmaps.items():
    bcmap['nummut'] = [len(x.split(',')) for x in bcmap['aa_mut']]
    print(k, bcmap['nummut'].value_counts(normalize=False))

plasmid nummut
1    524208
Name: count, dtype: int64
EO771_run1 nummut
1    62598
Name: count, dtype: int64
EO771_run2 nummut
1    195033
Name: count, dtype: int64
4T1_run1 nummut
1    71017
Name: count, dtype: int64
4T1_run2 nummut
1    198766
Name: count, dtype: int64


In [9]:
# This assigns position of observed mutation only for single mutations, and not for
# 'special' ones like STOP, RRDD, etc (Actually there arent any of these anymore, but there are still double muts)
for bcmap in bcmaps.values():
    bcmap['pos'] = -1
    f = (bcmap['nummut'] == 1) & (~bcmap['aa_mut'].isin(nonPositionalVars))
    if sum(~f) != 0:
        print('error') # these have been removed from map, shouldn't find anymore
    bcmap.loc[f,'pos'] = [int(x[1:-1]) for x in bcmap.loc[f,'aa_mut']]

## Combine pacbio maps

In [10]:
bcmaps.keys()

dict_keys(['plasmid', 'EO771_run1', 'EO771_run2', '4T1_run1', '4T1_run2'])

In [11]:
for celltype in ['EO771', '4T1']:
    bcmap = bcmaps['plasmid'][['aa_mut', 'score', 'count', 'closest_variant', 'vardist']].rename(
        columns={'aa_mut':'aa_mut_plasmid','score':'score_plasmid'}).copy()
    for run in ['run1', 'run2']:
        bcmap = bcmap.merge(bcmaps[f'{celltype}_{run}'][['aa_mut', 'score']].add_suffix(f'_{run}'), left_index=True, right_index=True, how='left')

    # set bcmaps['aa_mut_run1X'] to positional version of mutation (and same for run2)
    for c in ['plasmid', 'run1', 'run2']:
        c1 = f'aa_mut_{c}'
        c2 = c1 + 'X'
        bcmap[c2] = ''
        f = ~pd.isna(bcmap[c1])
        bcmap.loc[f,c2] = [x[:-1] + 'X' for x in bcmap.loc[f,c1]]

    # Store consensus values here
    bcmap['aa_mut'] = ''
    bcmap['score'] = -1.0
    bcmap['category'] = ''

    f1 = ~pd.isna(bcmap['aa_mut_run1'])
    f2 = ~pd.isna(bcmap['aa_mut_run2'])
    
    ## all three are present
    f = f1 & f2
    ### If all agree, use that
    g = (f) & (bcmap['aa_mut_run1'] == bcmap['aa_mut_run2']) & (bcmap['aa_mut_plasmid'] == bcmap['aa_mut_run1'])
    bcmap.loc[g, 'category'] = 'all agree'
    bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score', 'score_run1', 'score_run2']], axis=1)
    bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_plasmid']

    ### all three present, agree on position
    g = (f) & (bcmap['category'] == '') & (bcmap['aa_mut_run1X'] == bcmap['aa_mut_run2X']) & (bcmap['aa_mut_plasmidX'] == bcmap['aa_mut_run1X'])
    bcmap.loc[g, 'category'] = 'all agree pos'
    bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score', 'score_run1', 'score_run2']], axis=1)
    bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_plasmidX']

    ### run1 and run2 agree, use that 
    g = (f) & (bcmap['category'] == '') & (bcmap['aa_mut_run1'] == bcmap['aa_mut_run2']) 
    bcmap.loc[g,'category'] = 'run1/run2 agree, plasmid different'
    bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_run1']
    bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score_run1', 'score_run2']], axis=1)

    ### run1 and run2 agree on position
    g = (f) & (bcmap['category'] == '') & (bcmap['aa_mut_run1X'] == bcmap['aa_mut_run2X']) 
    bcmap.loc[g,'category'] = 'run1/run2 agree pos, plasmid different'
    bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_run1X']
    bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score_run1', 'score_run2']], axis=1)

    ### one of the runs agrees with plasmid, use that
    for run in ['run1', 'run2']:
        g = (f) & (bcmap['category'] == '') & (bcmap[f'aa_mut_{run}'] == bcmap['aa_mut_plasmid'])
        bcmap.loc[g,'category'] = f'{run}/plasmid agree'
        bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score_plasmid', f'score_{run}']], axis=1)
        bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_plasmid']
    
    ### one of the runs agrees with plasmid (pos), use that
    for run in ['run1', 'run2']:
        g = (f) & (bcmap['category'] == '') & (bcmap[f'aa_mut_{run}X'] == bcmap['aa_mut_plasmidX'])
        bcmap.loc[g,'category'] = f'{run}/plasmid agree pos'
        bcmap.loc[g,'score'] = np.max(bcmap.loc[g,['score_plasmid', f'score_{run}']], axis=1)
        bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_plasmidX']

    ### Otherwise we have 3 different calls... should be rare. Use highest scoring among cells
    g = (f) & (bcmap['category']=='')
    g1 = (g) & (bcmap['score_run1'] >= bcmap['score_run2'])
    bcmap.loc[g1,'category']='all disagree, use highest scoring (run1)'
    bcmap.loc[g1,'aa_mut'] = bcmap.loc[g1,'aa_mut_run1']
    bcmap.loc[g1,'score'] = bcmap.loc[g1,'score_run1']
    g1 = (g) & (bcmap['score_run2'] >  bcmap['score_run1'])
    bcmap.loc[g1,'category']='all disagree, use highest scoring (run2)'
    bcmap.loc[g1,'aa_mut'] = bcmap.loc[g1,'aa_mut_run2']
    bcmap.loc[g1,'score'] = bcmap.loc[g1,'score_run2']

    ## Only two are present. (Plasmid is always present). Always use the one from cell line
    for r1 in ['run1', 'run2']:
        if r1=='run1':
            r2='run2'
        else:
            r2='run1'
        c1 = f'aa_mut_{r1}'
        c2 = f'aa_mut_{r2}'
        g = (~pd.isna(bcmap[c1])) & (pd.isna(bcmap[c2]))
        g1 = (g) & (bcmap[c1] == bcmap['aa_mut_plasmid'])
        bcmap.loc[g1,'score'] = np.max(bcmap.loc[g1,[f'score_{r1}', 'score_plasmid']], axis=1)
        bcmap.loc[g1,'category'] = f'{r1}/plasmid agree'
        bcmap.loc[g1,'aa_mut'] = bcmap.loc[g1,'aa_mut_plasmid']

        g1 = (g) & (bcmap['category'] == '') & (bcmap[f'{c1}X'] == bcmap['aa_mut_plasmidX'])
        bcmap.loc[g1,'score'] = np.max(bcmap.loc[g1,[f'score_{r1}', 'score_plasmid']], axis=1)
        bcmap.loc[g1,'category'] = f'{r1}/plasmid agree pos'
        bcmap.loc[g1,'aa_mut'] = bcmap.loc[g1,'aa_mut_plasmidX']

        g1 = (g) & (bcmap['category'] == '')
        bcmap.loc[g1,'aa_mut'] = bcmap.loc[g1,c1]
        bcmap.loc[g1,'score'] = bcmap.loc[g1,f'score_{r1}']
        bcmap.loc[g1,'category'] = f'{r1}/plasmid disagree, use {r1}'

    # only plasmid present
    g = (~f1) & (~f2)
    bcmap.loc[g,'category'] = 'plasmid only'
    bcmap.loc[g,'score'] = bcmap.loc[g,'score_plasmid']
    bcmap.loc[g,'aa_mut'] = bcmap.loc[g,'aa_mut_plasmid']

    # Set rough ordering for different categories. Category is more important than score
    
    # First condense some equivalent categories. Also rename some for consistency
    bcmap['category'] = bcmap['category'].astype(str)
    f = (bcmap['category'] == 'all disagree, use highest scoring (run1)') | (bcmap['category'] == 'all disagree, use highest scoring (run2)')
    bcmap.loc[f,'category'] = 'all disagree, use highest scoring'
    
    f = (bcmap['category'] == 'run1/plasmid disagree, use run1') | (bcmap['category'] == 'run2/plasmid disagree, use run2')
    bcmap.loc[f,'category'] = 'cell line/plasmid disagree, use cell line'
    
    f = (bcmap['category'] == 'run1/plasmid agree pos') | (bcmap['category'] == 'run2/plasmid agree pos')
    bcmap.loc[f,'category'] = 'cell line/plasmid agree pos'
    
    f = (bcmap['category'] == 'run1/plasmid agree') | (bcmap['category'] == 'run2/plasmid agree')
    bcmap.loc[f,'category'] = 'cell line/plasmid agree'
    
    f = bcmap['category'] == 'run1/run2 agree pos, plasmid different'
    bcmap.loc[f,'category'] = 'cell lines agree pos, plasmid different'
    
    f = bcmap['category'] == 'run1/run2 agree, plasmid different'
    bcmap.loc[f,'category'] = 'cell lines agree, plasmid different'
    
    bcmap['category'] = pd.Categorical(bcmap['category'], 
                                       categories=['no pacbio', 
                                                   'plasmid only',
                                                   'all disagree, use highest scoring',
                                                   'cell line/plasmid disagree, use cell line',
                                                   'cell lines agree pos, plasmid different',
                                                   'cell lines agree, plasmid different',
                                                   'cell line/plasmid agree pos', 
                                                   'all agree pos', 
                                                   'cell line/plasmid agree', 
                                                   'all agree'], ordered=True)
    if sum(pd.isna(bcmap['category'])) != 0:
        print('Error: got NA categories')
        break
        
    # score is primarily based on category. 
    bcmap['score'] = bcmap['score'] + 100*bcmap['category'].cat.codes.astype('float64')
    print('\n'+celltype)
    print(pd.DataFrame(bcmap['category'].value_counts()).merge(bcmap['category'].value_counts(normalize=True), left_index=True, right_index=True))
    bcmap[['aa_mut', 'score', 'count', 'closest_variant', 'vardist', 'category']].to_csv(f'{output_dir}/bcmap_{celltype}.txt.gz', compression='gzip', sep='\t', index=True)


EO771
                                            count  proportion
category                                                     
plasmid only                               496211    0.946592
all agree                                   13789    0.026304
cell line/plasmid agree                      6375    0.012161
cell lines agree, plasmid different          3611    0.006888
cell line/plasmid disagree, use cell line    2960    0.005647
cell line/plasmid agree pos                   624    0.001190
all agree pos                                 379    0.000723
all disagree, use highest scoring             221    0.000422
cell lines agree pos, plasmid different        38    0.000072
no pacbio                                       0    0.000000

4T1
                                            count  proportion
category                                                     
plasmid only                               489673    0.934120
all agree                                   18250    0.034